In [28]:
import sys
import json
import subprocess
from pathlib import Path

import pandas as pd

# Directory paths
ROOT_DIR = Path.cwd()
SRC_DIR = ROOT_DIR / "src"
DATA_DIR = ROOT_DIR / "data"


def get_json_files():
    return sorted(DATA_DIR.glob("*.json"), key=lambda p: p.stat().st_mtime, reverse=True)

In [29]:
# Run the scraper script in src/
scraper_files = sorted(SRC_DIR.glob("*.py"))
if not scraper_files:
    raise FileNotFoundError("No Python files found in 'src/'.")

for script in scraper_files:
    result = subprocess.run(
        [sys.executable, str(script)],
        capture_output=True,
        text=True
    )
    print(f"Ran: {script.name}")
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(f"Error: {result.stderr}")

# Read the latest JSON file as pandas DataFrame
json_files = get_json_files()
if not json_files:
    raise FileNotFoundError("No JSON file found in 'data/' after running scraper.")

json_path = json_files[0]
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

if isinstance(data, list):
    df = pd.json_normalize(data)
elif isinstance(data, dict):
    list_keys = [k for k, v in data.items() if isinstance(v, list) and (len(v) == 0 or all(isinstance(x, dict) for x in v))]
    df = pd.json_normalize(data[list_keys[0]]) if len(list_keys) == 1 else pd.json_normalize(data)
else:
    df = pd.DataFrame({"value": [data]})

print(f"Loaded: {json_path}")
df

Ran: edcs_scraper.py
[+] Connecting to EDCS API...
[+] Connected. Page size 500 works. Total records in EDCS: 542,291

[lookup] No changes in lookup dictionary — using existing file.

[+] Data files found. Checking for updates...
    Local records  : 542,290
    EDCS total     : 542,291
[+] 1 new records found. Scraping updates...

[+] Scraping from offset  : 542,290
[+] Total records in EDCS : 542,291
[+] Page size             : 500
[+] Press Ctrl+C anytime  — progress saved every page

  offset=542,790/542,291 | saved=      0 | 100.0% | ~0min left      
[✓] Records saved : 0
[✓] Last EDCS ID  : EDCS-85701225
[✓] Checkpoint deleted — full scrape complete!

[✓] JSON : /Users/sanojdoddapaneni/Documents/Projects/EDCS-Analytics/data/edcs_inscriptions.json
[✓] TSV  : /Users/sanojdoddapaneni/Documents/Projects/EDCS-Analytics/data/edcs_inscriptions.tsv
[✓] Lookup: /Users/sanojdoddapaneni/Documents/Projects/EDCS-Analytics/data/edcs_lookup.json



JSONDecodeError: Extra data: line 542293 column 1 (char 249696247)

In [27]:
target_id = "EDCS-41200324"
df.loc[df["edcs_id"] == target_id, ["edcs_id", "inscription_text"]]

,edcs_id,inscription_text
354639,EDCS-41200324,?


In [25]:
list(df.columns)

['edcs_id',
 'monument_id',
 'province',
 'place',
 'material',
 'dating',
 'image_count',
 'longitude',
 'latitude',
 'inscription_text',
 'inscription_text_original',
 'language',
 'category',
 'subcategory',
 'praenomen',
 'nomen',
 'cognomen',
 'not_before',
 'not_after',
 'references',
 'images',
 'links']

In [26]:
# Treat both NaN and blank strings as missing values
blank_mask = df.astype("string").apply(lambda s: s.str.strip() == "")
missing_mask = df.isna() | blank_mask
missing_counts = missing_mask.sum().sort_values(ascending=False)
missing_cols = missing_counts[missing_counts > 0]

print(f"Columns with missing values (NaN or blank): {len(missing_cols)}")
missing_cols

Columns with missing values (NaN or blank): 19


cognomen                     542290
nomen                        542290
praenomen                    542290
images                       432692
not_after                    324465
not_before                   324453
dating                       324453
inscription_text_original    324443
links                        258991
subcategory                  165190
category                      96472
material                      74668
latitude                      18267
longitude                     18267
references                     4356
place                          3827
province                       3827
inscription_text                 43
language                         24
dtype: Int64

In [12]:
df.drop(columns=['inscription_text_original'], inplace=True)

In [21]:
name_cols = ["praenomen", "nomen", "cognomen"]

name_missing = (
    df[name_cols].isna()
    | (df[name_cols].astype("string").apply(lambda s: s.str.strip() == ""))
)

pd.DataFrame({
    "missing_count": name_missing.sum(),
    "total_rows": len(df),
    "missing_pct": (name_missing.sum() / len(df) * 100).round(2)
})

,missing_count,total_rows,missing_pct
praenomen,542290,542290,100.0
nomen,542290,542290,100.0
cognomen,542290,542290,100.0
